In [1]:
import os
import pandas as pd

# List files in the current working directory
print("Current directory:", os.getcwd())
print("\nAvailable files:")

for file in os.listdir():
    print(file)



df = pd.read_csv("../data/Sample - Superstore.csv", encoding="cp1252")

print("Dataset loaded successfully")
print("Shape:", df.shape)

display(df.head())

Current directory: d:\consulting\Retail Profitability Turnaround\notebook

Available files:
01_load_data.ipynb
02_analyis.ipynb
03_Root-Cause Analysis of Profitability Gaps.ipynb
04_Product_Region_Profitability.ipynb
05_Discount_Product_Analysis.ipynb
06-Business_Opportunity_Prioritization.ipynb
07_Validation_and_Consolidation.ipynb
Dataset loaded successfully
Shape: (9994, 21)


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    "../data/Sample - Superstore.csv",
    encoding="cp1252"
)

target = df[
    df["Sub-Category"].isin(["Tables", "Bookcases"])
].copy()

# Check 1: Overall figures
print("Rows in dataset:", len(df))
print("Target records:", len(target))
print("Target sales:", round(target["Sales"].sum(), 2))
print("Target net profit:", round(target["Profit"].sum(), 2))
print("Target profit margin (%):",
      round(target["Profit"].sum() / target["Sales"].sum() * 100, 2))
print("Loss-making records:", int((target["Profit"] < 0).sum()))

# Check 2: Subcategory totals
display(
    target.groupby("Sub-Category")
    .agg(
        Sales=("Sales", "sum"),
        Profit=("Profit", "sum"),
        Records=("Profit", "size"),
        Loss_Making_Records=("Profit", lambda x: (x < 0).sum())
    )
    .assign(
        Profit_Margin_Pct=lambda x: x["Profit"] / x["Sales"] * 100
    )
    .round(2)
)

# Check 3: Confirm regional totals reconcile to subcategory totals
regional_totals = target.groupby("Region")["Profit"].sum().sum()
print("Regional profit total:", round(regional_totals, 2))
print("Overall target profit:", round(target["Profit"].sum(), 2))
print("Reconciles:", np.isclose(regional_totals, target["Profit"].sum()))

Rows in dataset: 9994
Target records: 547
Target sales: 321845.53
Target net profit: -21198.04
Target profit margin (%): -6.59
Loss-making records: 312


,Sales,Profit,Records,Loss_Making_Records,Profit_Margin_Pct
Sub-Category,,,,,
Bookcases,114880.00,-3472.56,228,109,-3.02
Tables,206965.53,-17725.48,319,203,-8.56


Regional profit total: -21198.04
Overall target profit: -21198.04
Reconciles: True


### Validation and Consolidation of Findings

**Objective:** Validate the principal profitability metrics and consolidate the evidence developed throughout the root-cause analysis.

**Validation results**

The analysis used 9,994 records from the Sample Superstore dataset. Tables and Bookcases accounted for 547 records and $321,845.53 in sales. Their combined net profit was −$21,198.04, equivalent to a profit margin of −6.59%. Of the 547 records, 312 (57.04%) were loss-making.

Regional profits reconciled exactly to the overall net profit of −$21,198.04, confirming consistency between the regional and aggregate calculations.

**Consolidated findings**

1. **Tables are the primary profitability concern**, with a net loss of $17,725.48 and a margin of −8.56%.
2. **Bookcases are also unprofitable**, with a net loss of $3,472.56 and a margin of −3.02%.
3. **Tables in the East are the highest-priority segment for investigation**, recording a loss of $11,025.38 and a margin of −28.17%.
4. **Higher discount levels are associated with deteriorating profitability** in both subcategories. Tables show negative aggregate profit at the observed 20% discount level, while Bookcases approach break-even at that level.
5. **Losses are distributed across multiple products**, although the five worst-performing products account for approximately 29.1% of the negative profit summed across loss-making products.

**Conclusion:** The evidence supports prioritizing Table profitability, especially in the East, and reviewing discount practices and product-specific economics. These findings are descriptive: they do not establish causality or quantify recoverable savings. Any proposed intervention should be evaluated against potential changes in sales volume, costs, and product mix.